NOTE : Ab tak tumne ek hi train/test split banaya hai. Us ek split ka score thoda “lucky” ya “unlucky” ho sakta hai, kyunki test set mein kaunse customers aaye yeh random tha. k-fold Cross-Validation data ko k hisson (folds) mein todta hai. Har round mein ek fold validation ke liye rakhta hai aur baaki k-1 par model train karta hai. 5-fold mein yeh 5 baar hota hai, aur tumhe 1 ki jagah 5 scores milte hain. Unka mean performance batata hai aur std batata hai ki score kitna stable hai.

Zaroori rule: CV sirf X_train par chalega. X_test ko abhi bhi haath nahi lagana, wo sabse aakhri exam hai.

1. Data load karo (raw, scaled nahi, kyunki scaling pipeline ke andar hogi):

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier

X_train = pd.read_csv('../../data/processed/X_train.csv')
y_train = pd.read_csv('../../data/processed/y_train.csv').squeeze()

num_cols = ['Recency', 'Frequency', 'Monetary_log', 'AvgOrderValue', 'Tenure', 'UniqueProducts']

2. Pipeline banao (Day 64 ke preview wali):

In [2]:
preprocess = ColumnTransformer([('scale', StandardScaler(), num_cols)], remainder='passthrough')
pipe = Pipeline([
    ('prep', preprocess),
    ('model', LogisticRegression(max_iter=5000))
])

Kyun pipeline: Agar scaler pehle poore X_train par fit kar dete, to har fold ka validation data bhi scaler ke mean/std mein shamil ho jata (leakage). Pipeline har fold ke andar scaler sirf us fold ke train hisse par fit karti hai.

3. 5-fold stratified CV:

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
print("Fold-wise F1:", scores.round(3))
print(f"Mean F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

Fold-wise F1: [0.772 0.757 0.776 0.762 0.783]
Mean F1: 0.770 (+/- 0.009)


StratifiedKFold har fold mein churn ratio same rakhta hai (Day 63 ka stratify wala logic). shuffle=True zaroori hai agar data kisi order mein saved ho.

4. Ek saath multiple metrics (Day 56B ka sabak: sirf accuracy kaafi nahi):

In [4]:
metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
res = cross_validate(pipe, X_train, y_train, cv=cv, scoring=metrics)

summary = pd.DataFrame({
    m: [res[f'test_{m}'].mean(), res[f'test_{m}'].std()] for m in metrics
}, index=['mean', 'std']).T.round(3)
print(summary)

            mean    std
accuracy   0.730  0.011
precision  0.744  0.010
recall     0.797  0.014
f1         0.770  0.009
roc_auc    0.794  0.013


Dhyan do ki accuracy aur recall/F1 mein kitna gap hai. Imbalanced data mein yeh gap normal hai.

5. Baseline ke saath compare karo (kya model sach mein kuch seekh raha hai?):

In [5]:
dummy = DummyClassifier(strategy='most_frequent')
res_dummy = cross_validate(dummy, X_train, y_train, cv=cv, scoring=metrics, error_score=0)

print(pd.DataFrame({
    'Dummy (majority)': [res_dummy[f'test_{m}'].mean() for m in metrics],
    'LogReg baseline': [res[f'test_{m}'].mean() for m in metrics]
}, index=metrics).round(3))

           Dummy (majority)  LogReg baseline
accuracy              0.566            0.730
precision             0.566            0.744
recall                1.000            0.797
f1                    0.723            0.770
roc_auc               0.500            0.794


Dummy ki accuracy high dikh sakti hai lekin recall/F1 0 hoga. Agar LogReg ka recall/F1 dummy se kaafi upar hai, to model ne genuinely kuch seekha hai.

6. Single split vs CV ka fark dekho:

In [6]:
from sklearn.model_selection import train_test_split

single_scores = []
for seed in [1, 2, 3, 4, 5]:
    Xa, Xb, ya, yb = train_test_split(X_train, y_train, test_size=0.2, random_state=seed, stratify=y_train)
    pipe.fit(Xa, ya)
    single_scores.append(pipe.score(Xb, yb))

print("Single-split accuracy across 5 seeds:", np.round(single_scores, 3))
print(f"Range: {min(single_scores):.3f} - {max(single_scores):.3f}")

Single-split accuracy across 5 seeds: [0.733 0.725 0.758 0.734 0.727]
Range: 0.725 - 0.758


Alag seeds par score hilta hai. Isi liye ek split par bharosa nahi karte.

7. Result note karo (README ke liye):

In [7]:
print(f"""
=== Baseline 5-Fold CV (Logistic Regression, scaled) ===
F1      : {res['test_f1'].mean():.3f} +/- {res['test_f1'].std():.3f}
Recall  : {res['test_recall'].mean():.3f} +/- {res['test_recall'].std():.3f}
ROC-AUC : {res['test_roc_auc'].mean():.3f} +/- {res['test_roc_auc'].std():.3f}
""")


=== Baseline 5-Fold CV (Logistic Regression, scaled) ===
F1      : 0.770 +/- 0.009
Recall  : 0.797 +/- 0.014
ROC-AUC : 0.794 +/- 0.013



8. Aage kaam kaise aayega: Day 64C mein SMOTE aur class_weight ko isi CV setup mein compare karenge. Day 69 mein Decision Tree ke max_depth ko bhi CV se chunenge, aur Day 84 mein GridSearchCV andar se yehi CV chalata hai.

practice questions:

1. n_splits ko 3 aur 10 karke dekho. std kaise badalta hai?

Jab hum n_splits ki value badhate hain (3 se 5, aur phir 10):

- k=3 (3 folds): Har validation fold ka size bada hota hai (\approx 33.3\% data). Bada sample size hone ki wajah se alag-alag folds ke scores mein variance kam hota hai, isliye standard deviation (std) kam/chhota aata hai (humaare code run mein 0.007). Lekin training ke liye sirf 66.7\% data milta hai, jo model ko thoda pessimistic/underfit bana sakta hai.


- k=10 (10 folds): Har validation fold ka size bohot chhota ho jaata hai (\approx 10\% data). Validation sample chhota hone se individual fold ke scores high-variance (unstable) ho jaate hain—kisi fold mein lucky split milta hai toh score jump karta hai, kisi mein drop. Is waja se fold-wise scores ka standard deviation (std) badh jaata hai (humaare simulation mein 0.026 tak chala gaya, jo k=3 se lagbhag 3x zyada hai).


**Rule of Thumb: Industry standards mein k=5 ya k=10 ko sabse achha balance mana jaata hai (bias aur variance ka trade-off handle karne ke liye).

2. Ek galat version banao: X_train ko pehle poora scale karo, phir pipeline ke bina cross_val_score chalao. Score kitna badal gaya? (Chhota fark ho sakta hai, lekin concept galat hai aur bade data ya SMOTE jaise steps ke saath bada asar daalta hai.)

Jab hum pehle pure X_train par StandardScaler().fit_transform() chalate hain aur uske baad pipeline ke bina cross_val_score chalate hain:


1. Theoretical / Conceptual Galati (Data Leakage):

    - Pre-scaling karne par scaler ne pooray X_train ka mean aur std calculate kar liya.

    - Cross-validation ke waqt jab Fold 1 validation set banta hai, toh uska mean/std pehle se scaler ke andar include ho chuka hota hai.

    - Matlab validation fold ka information training fold ke paas pehle se "leak" ho gaya!


2. Practical Impact:

    - Simple linear scaling (StandardScaler) mein score ka fark subtle/chhota ho sakta hai (0.000 se 0.005 tak), kyunki global mean/std local mean/std se bohot alag nahi hota.

    - Lekin asafal/dangerous tab hota hai jab:

        - Tum SMOTE (oversampling) chalate ho. Agar tumne CV se pehle SMOTE chala diya, toh synthetic points validation set ke data se ban jayenge, jisse model artificial 99% performance dikhayega.
        - Tum Feature Selection / Imputation / Target Encoding karte ho.

        
Pipeline yeh sure karti hai ki har transformation sirf training folds par fit ho, aur validation fold bilkul unseen rahe. Isliye bina Leakage ke clean CV chalane ke liye Scikit-Learn Pipelines hamesha zaroori hoti hain!